# KNN (K Nearest Neighbors)

In [ ]:
'''
- Lazy Algorithm
- Find number of Optimal Neighbors (K)
    -> Elbow Method
        -> Error Rate (Lowest Point) or Accuracy Score (Highest Point)
        
        -> Calculate Error Rate
            => Total Wrong Prediction/ Total Test Data

model.fit(X_train, Y_train)
model.predict(X_test)

k_range = range(1, 21)
best_range (3 to 7)
considerable range (3 to 12)

'''

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import KNN Model
from sklearn.neighbors import KNeighborsClassifier
# Import Train Test Split
from sklearn.model_selection import train_test_split
# Import Data Preprocessing Module
from sklearn.preprocessing import StandardScaler, LabelEncoder
# Import Metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [6]:
df = pd.read_csv('D:\Data Science\Dataset\student.csv')
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Acer\AppData\Local\Temp\ipykernel_19860\3770232234.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv('D:\Data Science\Dataset\student.csv')


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,higher education,higher education,at_home,teacher,...,very good,medium,high,very low,very low,good,6,5,6,6
1,GP,F,17,U,GT3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,excellent,medium,medium,very low,very low,good,4,5,5,6
2,GP,F,15,U,LE3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,very good,medium,low,low,medium,good,10,7,8,10
3,GP,F,15,U,GT3,T,higher education,5th to 9th grade,health,services,...,good,low,low,very low,very low,excellent,2,15,14,15
4,GP,F,16,U,GT3,T,secondary education,secondary education,other,other,...,very good,medium,low,very low,low,excellent,4,6,10,10


## Continuous Data to Category

In [7]:
df['G3'].sort_values(ascending=True).unique()

array([ 0,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
       20], dtype=int64)

In [8]:
def convertMarks(marks):
    if marks >= 15:
        return 'A'
    elif marks >=11:
        return 'B'
    elif marks >=7:
        return 'C'
    else:
        return 'Fail'

In [10]:
df['Grade'] = df['G3'].apply(convertMarks)

In [11]:
df

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,Grade
0,GP,F,18,U,GT3,A,higher education,higher education,at_home,teacher,...,medium,high,very low,very low,good,6,5,6,6,Fail
1,GP,F,17,U,GT3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,medium,medium,very low,very low,good,4,5,5,6,Fail
2,GP,F,15,U,LE3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,medium,low,low,medium,good,10,7,8,10,C
3,GP,F,15,U,GT3,T,higher education,5th to 9th grade,health,services,...,low,low,very low,very low,excellent,2,15,14,15,A
4,GP,F,16,U,GT3,T,secondary education,secondary education,other,other,...,medium,low,very low,low,excellent,4,6,10,10,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,MS,M,20,U,LE3,A,5th to 9th grade,5th to 9th grade,services,services,...,very high,high,high,very high,very good,11,9,9,9,C
391,MS,M,17,U,LE3,T,secondary education,primary education (4th grade),services,services,...,high,very high,medium,high,bad,3,14,16,16,A
392,MS,M,21,R,GT3,T,primary education (4th grade),primary education (4th grade),other,other,...,very high,medium,medium,medium,good,3,10,8,7,C
393,MS,M,18,R,LE3,T,secondary education,5th to 9th grade,services,other,...,high,very low,medium,high,excellent,0,11,12,10,C


In [12]:
df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'Grade'],
      dtype='object')

In [ ]:
columns = 'Medu', 'Fedu','traveltime', 'studytime','failures', 'schoolsup', 'famsup','nursery','internet','freetime', 'goout', 'Dalc',
'Walc', 'health', 'absences', 'G1', 'G2', 'Grade'

le_columns = 'schoolsup', 'famsup','nursery','internet'

## Feature Encoding

In [14]:
# Label Encoder
le = LabelEncoder()
def encodeData(dataframe, *args):
    for col in args:
        dataframe[f'{col}_id'] = le.fit_transform(dataframe[col])
    print('Columns encoded.')
encodeData(df,'schoolsup', 'famsup','nursery','internet')

Columns encoded.


In [35]:
df['health'].unique()

array(['good', 'excellent', 'very bad', 'bad', 'very good'], dtype=object)

In [22]:
val = ['none','primary education (4th grade)','5th to 9th grade',
       'secondary education', 'higher education', ]

def manualEncode(dataframe, col, *args):
    encode_val = {}
    for idx, val in enumerate(args):
        encode_val[val]=idx
    dataframe[f'{col}_id'] = dataframe[col].map(encode_val)

In [23]:
manualEncode(df, 'Fedu','none','primary education (4th grade)','5th to 9th grade',
       'secondary education', 'higher education')

In [24]:
manualEncode(df, 'Medu','none','primary education (4th grade)','5th to 9th grade',
       'secondary education', 'higher education')

In [27]:
manualEncode(df, 'traveltime','<15 min.', '15 to 30 min.', '30 min. to 1 hour', '>1 hour')

In [29]:
manualEncode(df, 'studytime','<2 hours', '2 to 5 hours', '5 to 10 hours', '>10 hours')

In [31]:
manualEncode(df, 'freetime','very low','low','medium','low', 'high',  'very high' )
manualEncode(df, 'goout','very low','low','medium','low', 'high',  'very high' )
manualEncode(df, 'Dalc','very low','low','medium','low', 'high',  'very high' )
manualEncode(df, 'Walc','very low','low','medium','low', 'high',  'very high' )

In [36]:
manualEncode(df, 'Grade','Fail','C','B','A')
manualEncode(df, 'health','very bad', 'bad', 'good', 'very good', 'excellent')

In [37]:
df

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,Fedu_id,Medu_id,traveltime_id,studytime_id,freetime_id,goout_id,Dalc_id,Walc_id,Grade_id,health_id
0,GP,F,18,U,GT3,A,higher education,higher education,at_home,teacher,...,4,4,1,1,2,4,0,0,0,2
1,GP,F,17,U,GT3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,1,1,0,1,2,2,0,0,0,2
2,GP,F,15,U,LE3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,1,1,0,1,2,3,3,2,1,2
3,GP,F,15,U,GT3,T,higher education,5th to 9th grade,health,services,...,2,4,0,2,3,3,0,0,3,4
4,GP,F,16,U,GT3,T,secondary education,secondary education,other,other,...,3,3,0,1,2,3,0,3,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,MS,M,20,U,LE3,A,5th to 9th grade,5th to 9th grade,services,services,...,2,2,0,1,5,4,4,5,1,3
391,MS,M,17,U,LE3,T,secondary education,primary education (4th grade),services,services,...,1,3,1,0,4,5,2,4,3,1
392,MS,M,21,R,GT3,T,primary education (4th grade),primary education (4th grade),other,other,...,1,1,0,0,5,2,2,2,1,2
393,MS,M,18,R,LE3,T,secondary education,5th to 9th grade,services,other,...,2,3,2,0,4,0,2,4,1,4


In [38]:
df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'Grade', 'schoolsup_id',
       'famsup_id', 'nursery_id', 'internet_id', 'Fedu_id', 'Medu_id',
       'traveltime_id', 'studytime_id', 'freetime_id', 'goout_id', 'Dalc_id',
       'Walc_id', 'Grade_id', 'health_id'],
      dtype='object')

## Features and Target

In [40]:
features = ['absences', 'G1', 'G2','schoolsup_id','famsup_id', 
'nursery_id', 'internet_id', 'Fedu_id', 'Medu_id','traveltime_id',
'studytime_id', 'freetime_id', 'goout_id', 'Dalc_id',
'Walc_id','failures','health_id']
target = 'Grade_id'

In [41]:
X = df[features]
Y = df[target]

## Train Test Split

In [42]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

## Feature Scaling

In [43]:
scaler = StandardScaler()
X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)